# Jupyter notebooks

If you've not used Jupyter notebooks before, here are some tips:

- A notebook is made of of several cells. Each cell is either made up of text (like this one) or code (the one which imports spikeinterface, below).
- You can add a new cell of your own by clicking the + button on the last row of the menu above.
- You can execute the code in a cell by pressing shift+Enter or ctrl+Enter (cmd+Enter on a mac). Whatever you run will be "remembered" by the Python kernel. So if you assign `a=3` in one cell, you can use `a` later on.
- While reading this notebook, read the text then execute the code you find in each cell.

Some people (me!) hate Jupyter because your results can depend on the order that you execute cells - this means it's easy to do stupid stuff. Be careful!


In [ ]:
import spikeinterface.full as si
from pathlib import Path

# 1. Load a recording

We will use a recording from DANDI.

DANDI (https://dandiarchive.org/) is a _huge_ repository, with a lot of ephys data on it. We'll look at a dataset from Adrian: https://dandiarchive.org/dandiset/000939/

That dataset has a lot of subjects and sessions in it. We'll pick this session: https://dandiarchive.org/dandiset/000939/0.260512.1701/files?location=sub-A3702&page=1

You can either download the 26GB file to your laptop. Or stream parts of it directly from DANDI. Go to Section 1.1 if you want to steam it. Go to Section 1.2 if you've downloaded the file locally and want to load it. I've saved it in the `data` folder in this repository.


## 1.1 Stream the recording

To stream the specific file we want, we need to find where the data is actually stored. We can find this out by investigating the metadata of the session, either by clicking "metadata" on the webpage or clicking here: https://api.dandiarchive.org/api/dandisets/000939/versions/0.260512.1701/assets/6fe925dd-ea41-4a84-9cee-ca6bb039edcb/

Alternatively, you could use the [dandi python package](https://pynwb.readthedocs.io/en/stable/tutorials/advanced_io/streaming.html) , or [loadi](https://github.com/chrishalcrow/loadi) (which I made). Both help you to load things without having to go get these annoying urls.


In [ ]:
# stream the recording - this will take ~1min, as it's loading some data about the recording
recording = si.read_nwb_recording(
    file_path = 'https://dandiarchive.s3.amazonaws.com/blobs/a8f/800/a8f8003e-4483-4b50-8a45-91ac5971f5d5',
    stream_mode = 'fsspec',
    electrical_series_path = 'acquisition/ElectricalSeries',
)


In [ ]:
recording

## 1.2 Download, then load the recording locally

This is more similar to what you'd do with your own data. Just point the `file_path` parameter at your download location


In [ ]:
# EDIT THIS FILE PATH. I've saved the file in the `data` folder of this repository
recording_path = Path('data/sub-A3702_ses-191126_behavior+ecephys.nwb')

recording = si.read_nwb_recording(
    file_path = recording_path,
    electrical_series_path = 'acquisition/ElectricalSeries',
)


## 1.3 Exercise: load your own data

You can load lots of different types of data into SpikeInterface. Here's a list: https://spikeinterface.readthedocs.io/en/stable/modules/extractors.html#raw-data-formats

In my lab, we use openephys. So we use `si.read_openephys('path/to/recording')`.

Try to load some of your own data


In [ ]:
my_own_data = si.read_...



# 2. Play with the recording


We now have the `recording` object. We can extract useful properties from it, like the channel locations, sampling frequency...


In [ ]:
print(f"The samping frequency is {recording.sampling_frequency}")

channel_locations = recording.get_channel_locations()
y_locations = channel_locations[:,1]
max_y_location = max(y_locations)
print(f"The max y location of a channel is {max_y_location}. This is measured in microns.")
print(f"The recording start time is {recording.get_time_info()['t_start']}")


In [ ]:
# You can see what else there is to explore by typing out "recording." and pressing the tab bar:
rec


And you can slice it up, using time or channels


In [ ]:
# Take the first 5 minutes of the recording, and the first 4 channels
recording_slice = recording.time_slice(start_time=0, end_time=60*5).select_channels(channel_ids=recording.channel_ids[:4])


And visualize the raw trace


In [ ]:
%matplotlib widget
si.plot_traces(recording_slice, backend='ipywidgets')


But don't be scared of plotting a full giant recording. SpikeInterface uses a _lazy_ loading system. It doesn't load the entire 26GB recording. Instead, it only loads the metadata and will then load the raw traces when you need them


In [ ]:
%matplotlib widget
si.plot_traces(recording, backend='ipywidgets', mode='map')


Hmmmm... That looks gross. We need to...

# 3. Preprocess

It's quite hard to see spikes in raw ephys data, because global eletrical signal dominate. To see them we need to _at least_ bandpass filter. On a high density probe we would normally do a common reference too.

In SpikeInterface we think of preprocessing as a chain of steps. It's easiest just to see this in action:


In [ ]:
preprocessed_recording = si.common_reference(si.bandpass_filter(si.depth_order(recording)))


In [ ]:
# let's take a look...
si.plot_traces(recording_slice, backend='ipywidgets', mode = 'map')


If you're happy, you can skip to Section 4

# 3.1 Exercise: Try out some other preprocessing steps

Here's a list: https://spikeinterface.readthedocs.io/en/latest/api.html#api-preprocessing

Maybe try out a normalize, or a whiten, and see if you can still find spikes in the recording.

# 3.2 Exercise: Find some bad channels

From the visualisation, it seems like some channels are bad - probably the electrode on the device is dead. Try out the `si.detect_bad_channels` function to see if you can find them

# 3.3 Exercise: Write a preprocessing pipeline

Once you like your steps, try writing a preprocessing pipleine, as it detailed here: https://spikeinterface.readthedocs.io/en/latest/how_to/build_pipeline_with_dicts.html


## 3.4 Motion correction (optional)

Especially in acute recordings, probe motion is a big problem for spike sorting.

At the moment the most common approach to deal with motion is to 1) estimate the motion 2) try to correct the motion by re-sampling and interpolating the raw recording. There's mounting evidence that we're ok at estimating motion but we're bad at interpolating.

To estimate the motion we can use the `si.compute_motion` function, and then use `si.plot_motion` to visualise the result.

If you want to interpolate, you can then use `si.interpolate

We'll do this below for a 5 minute slice of the recording use the fastest method, called `rigid_fast`. Read more about motion correction here: https://spikeinterface.readthedocs.io/en/latest/how_to/handle_drift.html



In [ ]:
# Note: this will take 5-10mins to run...
motion, motion_info = si.compute_motion(preprocessed_recording.time_slice(0, 5*60), preset='nonrigid_fast_and_accurate', output_motion_info=True)


In [ ]:
si.plot_motion(motion)

In this example, for the first 5 mins and for the simplest motion computation, the motion is _much less_ than the spacing between two electrodes. Hence, I would recommend not doing any motion correction. For real data you should check the entire recording. I recommend the DREDGE preset.


# 4. Sort

We now have a preprocessed recording, so are ready to sort it. SpikeInterface supports many sorters, see a list here: https://spikeinterface.readthedocs.io/en/stable/modules/sorters.html#supported-spike-sorters.
We'll try an run a new sorter called LUPIN (the gentleman thief - read more here https://www.biorxiv.org/content/10.64898/2026.01.23.701239v2). To do this we simply run `si.run_sorter`.

For the sake of time, we'll just sort the first five minutes of the recording. In reality, you should sort your full recording. Note that sorters usually except longer recordings. In my experience (working with chronic NeuroPixel probes), 20 mins is usually ok.

In [ ]:
# This (roughly) sets the number of cores to use when sorting
si.set_global_job_kwargs(n_jobs=4)

recording_for_sorting = preprocessed_recording.time_slice(start_time=0, end_time=5*60)
sorting_lupin = si.run_sorter(
    sorter_name='lupin',
    recording = recording_for_sorting,
    verbose=True,
)

# it's a hat...

To use a different sorter, e.g. kilosort or mountainsort, you can swap "lupin" with "kilosort4" or "mountainsort5". You'll probably need to install these sorters in your environment. If you're using `uv` just do e.g. `uv add kilosort` in your project folder. If you're using a venv, do e.g. `pip install kilosort4`.

Now that we have a sorting. We can take a look at some of it's properties:

In [ ]:
print(f"This sorting found {sorting_lupin.get_num_units()} units")

Try typing `sorting_lupin` below, then pressing tab to see what you can do with the output:

## 4.1 Exercise: run kilosort

Install and run kilosort to make another sorting

To get to the fun stuff we need to do some...

# 5. PostProcessing

In postprocessing, we compute data based on the recording and sorting. This is the stuff you're really interested in: average waveforms (templates), spike-amplitude and -depth distributions, PCAs and metrics like snr, isi_violations and decay rates.

In SpikeInterface we compute these things using a `SortingAnalyzer`. This is **the most important** object in SpikeInterface. It's the thing a full sorting pipeline will output. You'll use this to investigate your results, curate units and export results to other packages (like `Pynapple`). An analyzer is basically a combined recording+sorting object. To make one, you just need to combine these two other objects together. When creating it, I'm also going to tell SpikeInterface to save the analyzer in a folder, and where to save it.


In [ ]:
analyzer_lupin = si.create_sorting_analyzer(
    sorting=sorting_lupin,
    recording=recording_for_sorting,
    format='binary_folder',
    folder='data/lupin_analyzer_adrian'
)

If you wanted to compare "lupin" to another sorter, you'd make another analyzer using the same recording but a different "kilosort" sorting, and use our comparison tools.

Now that we have an analyzer, we can compute _extensions_. Extensions contain the information you want to know. Read more about them here: https://spikeinterface.readthedocs.io/en/stable/modules/postprocessing.html#extensions-as-analyzerextensions

To start, let's compute the `noise_levels`.

In [ ]:
analyzer_lupin.compute('noise_levels')

Nice! Go look at the `lupin_analyzer_adrian` folder, and see if you can find where the `noise_levels` are saved.

We can check what other extensions we can compute as follows:

In [ ]:
analyzer_lupin.get_computable_extensions()

Let's try another one. We'll now compute the average waveform (template) for each unit:

In [ ]:
analyzer_lupin.compute('templates')

Oh no! That didn't work. This is because the `templates` extensions depends on the `waveform|random_spikes` extension. Roughly: some extensions depend on each other. The templates are the average waveforms so it makes sense that to compute them, you first need the waveforms. Read more about extensions and their dependencies here: 

At the end of the day, it's easiest to figure out which extensions you want to compute and compute them all at once. Like so:

In [ ]:
analyzer_lupin.compute([
    'random_spikes',
     'waveforms',
     'templates',
     'noise_levels',
     'correlograms',
     'amplitude_scalings',
     'spike_locations',
     'template_similarity',
     'unit_locations',
     'quality_metrics',
     'template_metrics',
])

With that, we've create an analyzer and computed loads of useful information about it.

We can visualise the analyzer by going back to the terminal and running

```
uv run sigui data/lupin_analyzer_adrian
```

# Big exercise

Open `sorting_script.py` and write a spike sorting pipeline from start to finish. Then run it. I've put some skeleton code in there.

# Next steps

This was a whistle stop tour of spike sorting. Here are some resources for further study:

- GitHub repo for a 2 day spike sorting course at the Human Technopole in Milan:
- SpikeInterface docs: 
- Curation
    - Bombcell
    - UnitRefine
- SpikeInterface-GUI, for viewing your outpout